# 🗄️ Notebook 02 — RAG Setup: Preprocessing & Vector Store Build

**Goal:** Transform raw datasets into a searchable ChromaDB vector store.

**Pipeline:**
```
PubMedQA + MedQuAD
    → clean_text()          (unicode, whitespace)
    → RecursiveCharacterTextSplitter  (chunk_size=900, overlap=175)
    → FastEmbedEmbeddings   (BAAI/bge-small-en-v1.5)
    → ChromaDB persist      (./chroma_medical_db)
```

⏱️ **Estimated time:** 15–45 min depending on hardware (CPU-only).
After the first run, use Cell 10 (reconnect) instead of rebuilding.

In [ ]:
# Cell 1: Install dependencies (run once)
# %pip install -q langchain langchain-community chromadb fastembed datasets tqdm

In [ ]:
# Cell 2: Imports & path setup
import sys, time
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from src.rag.loader import load_pubmedqa, load_medquad
from src.rag.preprocessor import preprocess_documents, clean_text
from src.rag.vectorstore import build_vectorstore, load_vectorstore
from src.utils import get_logger

logger = get_logger('02_rag_setup')
sns.set_theme(style='whitegrid')
print('✅ Imports OK')

In [ ]:
# Cell 3: Load datasets
t0 = time.time()
pubmed_docs = load_pubmedqa(limit=1000)
medquad_docs = load_medquad(limit=1000)
raw_docs = pubmed_docs + medquad_docs
print(f'Total raw docs: {len(raw_docs)} ({time.time()-t0:.1f}s)')

In [ ]:
# Cell 4: Demonstrate clean_text on a sample
sample = raw_docs[0].page_content
print('=== BEFORE clean_text ===')
print(repr(sample[:200]))
print()
print('=== AFTER clean_text ===')
print(repr(clean_text(sample[:200])))

In [ ]:
# Cell 5: Preprocess (clean + chunk)
t0 = time.time()
chunks = preprocess_documents(raw_docs)  # uses config.yaml defaults
elapsed = time.time() - t0
print(f'Produced {len(chunks):,} chunks in {elapsed:.1f}s')
print(f'Sample chunk:\n{chunks[0].page_content[:300]}\n...')
print(f'Metadata: {chunks[0].metadata}')

In [ ]:
# Cell 6: Chunk length distribution
chunk_lens = [len(c.page_content) for c in chunks]
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(chunk_lens, bins=50, kde=True, ax=ax, color='steelblue')
ax.axvline(pd.Series(chunk_lens).median(), color='red', linestyle='--',
           label=f'Median: {pd.Series(chunk_lens).median():.0f} chars')
ax.set_title('Chunk Length Distribution (chars)', fontsize=13, fontweight='bold')
ax.set_xlabel('Characters per chunk')
ax.legend()
plt.tight_layout()
plt.savefig('../data/results/06_chunk_length_dist.png', bbox_inches='tight')
plt.show()

In [ ]:
# Cell 7: Source breakdown of chunks
import collections
source_counts = collections.Counter(c.metadata.get('source', 'Unknown') for c in chunks)
for src, cnt in source_counts.items():
    print(f'  {src}: {cnt:,} chunks ({100*cnt/len(chunks):.1f}%)')

In [ ]:
# Cell 8: ⏱️ BUILD VECTORSTORE (run ONLY ONCE — takes 15-45 minutes on CPU)
# If you have already built it, skip to Cell 10.

print('⏳ Starting ChromaDB build. This may take a while...')
t0 = time.time()

vectorstore = build_vectorstore(chunks, batch_size=500)

elapsed_min = (time.time() - t0) / 60
print(f'\n✅ Done in {elapsed_min:.1f} min')
print(f'Total vectors: {vectorstore._collection.count():,}')

In [ ]:
# Cell 9: Test a query on the freshly built store
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
test_query = 'What are the symptoms of Type 2 Diabetes?'
docs = retriever.invoke(test_query)
print(f'Query: "{test_query}"')
print(f'Retrieved {len(docs)} chunks:')
for i, d in enumerate(docs, 1):
    print(f'\n[{i}] Source: {d.metadata.get("source")}\n{d.page_content[:200]}...')

In [ ]:
# Cell 10: ⚡ RECONNECT to existing vectorstore (fast — use this on subsequent runs)
vs = load_vectorstore()
count = vs._collection.count()
print(f'✅ Connected to existing vectorstore: {count:,} documents')

In [ ]:
# Cell 11: Retrieval latency benchmark (10 queries)
import numpy as np

test_queries = [
    'What is hypertension?',
    'symptoms of heart attack',
    'metformin side effects',
    'diabetes diet recommendations',
    'how does insulin work',
    'treatment for stroke',
    'blood pressure normal range',
    'cancer screening guidelines',
    'antibiotics resistance',
    'mental health depression treatment',
]

latencies = []
retriever_k5 = vs.as_retriever(search_kwargs={'k': 5})

for q in test_queries:
    t0 = time.time()
    _ = retriever_k5.invoke(q)
    latencies.append((time.time() - t0) * 1000)

print(f'Retrieval latency (k=5):')
print(f'  Mean:   {np.mean(latencies):.1f} ms')
print(f'  Median: {np.median(latencies):.1f} ms')
print(f'  Max:    {np.max(latencies):.1f} ms')
print(f'  Min:    {np.min(latencies):.1f} ms')